In [ ]:
# Locate the repository when Jupyter starts in Notebooks/.
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks': PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import SRC.KingCRAB.rga as rga_helpers
from SRC.KingCRAB.context import configure_module
from SRC.KingCRAB.rga import ROI, ReadFile, avg_background_intensity_at, compositions, find_data_start, highlight_windows, integrate_window, normalize_area, read_rga, subtract_background, xe_percent, xe_percent_series

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

In [ ]:
configure_module(rga_helpers, globals())

DataWarm        = ReadFile("../Data/WARM2.txt")
DataCold        = ReadFile("../Data/COLD13.txt")
Background      = ReadFile("../Data/BG.txt")


DataWarm   = DataWarm.rename(columns={"              Mass (AMU)": "Mass", "         Intensity (Torr)": "Intensity"})
DataCold   = DataCold.rename(columns={"              Mass (AMU)": "Mass", "         Intensity (Torr)": "Intensity"})
Background = Background.rename(columns={"              Mass (AMU)": "Mass", "         Intensity (Torr)": "Intensity"})


print(Background.columns)

In [ ]:
# Runs were taken at different absolute pressures in the RGA therefore we need the partial pressure

configure_module(rga_helpers, globals())

DataWarm_norm   = normalize_area(DataWarm)
DataCold_norm   = normalize_area(DataCold)
Background_norm = normalize_area(Background)

In [ ]:
# There is some background gas in the RGA and system, therefore we want to subtract that

configure_module(rga_helpers, globals())

DataWarm_BS  = subtract_background(DataWarm_norm, Background_norm)
DataCold_BS  = subtract_background(DataCold_norm, Background_norm)

In [ ]:
WINDOWS = {
    "Ar_Single":  (36, 41.0),  
    "Ar_Double":  (19.0, 21.0),  
    "Xe_Double":  (63.0, 70.0),
    "Xe_Single":  (126.0, 137.0),
}

configure_module(rga_helpers, globals())


# --- Optional: visualize ROIs on a spectrum ---

# Example usage:
# fig, ax = plt.subplots()
# ax.plot(df_bgsub["Mass"], df_bgsub["Intensity"])
# highlight_windows(ax, WINDOWS)
# plt.show()

In [ ]:
PartsWarm, FracWarm = compositions(DataWarm_BS, WINDOWS)
PartsCold, FracCold = compositions(DataCold_BS, WINDOWS)

# Warm data
XeWarm = PartsWarm["Xe_Single"] + PartsWarm["Xe_Double"]
ArWarm = PartsWarm["Ar_Single"] + PartsWarm["Ar_Double"]
TotWarm = XeWarm + ArWarm + PartsWarm["Other"]

print("Warm:")
print("  Xe_Single:", PartsWarm["Xe_Single"])
print("  Xe_Double:", PartsWarm["Xe_Double"])
print("  Ar_Single:", PartsWarm["Ar_Single"])
print("  Ar_Double:", PartsWarm["Ar_Double"])
print("  Other:", PartsWarm["Other"])
print("  Total Warm:", TotWarm)

# Cold data
XeCold = PartsCold["Xe_Single"] + PartsCold["Xe_Double"]
ArCold = PartsCold["Ar_Single"] + PartsCold["Ar_Double"]
TotCold = XeCold + ArCold + PartsCold["Other"]

print("\nCold:")
print("  Xe_Single:", PartsCold["Xe_Single"])
print("  Xe_Double:", PartsCold["Xe_Double"])
print("  Ar_Single:", PartsCold["Ar_Single"])
print("  Ar_Double:", PartsCold["Ar_Double"])
print("  Other:", PartsCold["Other"])
print("  Total Cold:", TotCold)

In [ ]:
# Sums for Ar
ArWarm = PartsWarm["Ar_Single"] + PartsWarm["Ar_Double"]
ArCold = PartsCold["Ar_Single"] + PartsCold["Ar_Double"]

# --- regular full fractions including Other ---
Frac_Xe_Warm  = 100 * XeWarm / TotWarm
Frac_Ar_Warm  = 100 * ArWarm / TotWarm
Frac_Oth_Warm = 100 * PartsWarm["Other"] / TotWarm

Frac_Xe_Cold  = 100 * XeCold / TotCold
Frac_Ar_Cold  = 100 * ArCold / TotCold
Frac_Oth_Cold = 100 * PartsCold["Other"] / TotCold

print("Before cooling (with Other):", Frac_Xe_Warm, Frac_Ar_Warm, Frac_Oth_Warm)
print("After  cooling (with Other):", Frac_Xe_Cold, Frac_Ar_Cold, Frac_Oth_Cold)

# --- Xe vs Ar only (ignore Other) ---
Frac_Xe_Warm_noOth = 100 * XeWarm / (XeWarm + ArWarm)
Frac_Ar_Warm_noOth = 100 - Frac_Xe_Warm_noOth

Frac_Xe_Cold_noOth = 100 * XeCold / (XeCold + ArCold)
Frac_Ar_Cold_noOth = 100 - Frac_Xe_Cold_noOth

print("\nBefore cooling (Xe vs Ar only): Xe =", Frac_Xe_Warm_noOth, "Ar =", Frac_Ar_Warm_noOth)
print("After  cooling (Xe vs Ar only):  Xe =", Frac_Xe_Cold_noOth, "Ar =", Frac_Ar_Cold_noOth)

In [ ]:
configure_module(rga_helpers, globals())




files = [
    "WARM.txt",
    "WARM2.txt",
    "COLD2.txt",
    "COLD3.txt",
    "COLD4.txt",
    "COLD5.txt",
    "COLD6.txt",
    "COLD7.txt",
    "COLD8.txt",
    "COLD9.txt",
    "COLD10.txt",
    "COLD11.txt",
    "COLD12.txt",
    "COLD13.txt",
]
names = [Path(f).stem for f in files]  
xe_df = xe_percent_series(files, Background, WINDOWS, names)

plt.figure(figsize=(9,5))
plt.bar(xe_df["Dataset"], xe_df["Xe_%"])
plt.ylabel("Xe % (Xe vs Ar only)")
plt.title("Xenon Fraction vs Datasets")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 10)
plt.tight_layout()
plt.show()

In [ ]:
# Example: values from compositions
XeWarm = PartsWarm["Xe_Single"] + PartsWarm["Xe_Double"]
XeCold = PartsCold["Xe_Single"] + PartsCold["Xe_Double"]
ArWarm = PartsWarm["Ar_Single"] + PartsWarm["Ar_Double"]
ArCold = PartsCold["Ar_Single"] + PartsCold["Ar_Double"]

# scale cold run so Ar matches warm
scale = ArWarm / ArCold if ArCold != 0 else 1.0
XeCold_scaled = XeCold * scale

# percent xenon captured
Xe_captured_percent = 100 * (XeWarm - XeCold_scaled) / XeWarm

print(f"Percent Xe captured: {Xe_captured_percent:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

COLORS = {
    "warm": "#D55E00",   # muted orange
    "cold": "#0072B2",   # muted blue
    "roi":  "#999999"    # neutral gray
}

plt.figure(figsize=(10, 6))

floor = 1e-6
warm_mask = DataWarm_BS["Intensity"] > floor
cold_mask = DataCold_BS["Intensity"] > floor

warm_y = DataWarm_BS["Intensity"].where(warm_mask, np.nan)
cold_y = DataCold_BS["Intensity"].where(cold_mask, np.nan)

# Main spectra (thicker lines)
plt.plot(
    DataWarm_BS["Mass"], warm_y,
    label="Warm", color=COLORS["warm"], lw=3
)
plt.plot(
    DataCold_BS["Mass"], cold_y,
    label="Cold", color=COLORS["cold"], lw=3
)

# ROI shading
for i, (name, (lo, hi)) in enumerate(WINDOWS.items()):
    plt.axvspan(
        lo, hi,
        color=COLORS["roi"],
        alpha=0.08,
        label="ROI" if i == 0 else None
    )

plt.yscale("log")

# Bigger labels + spacing
plt.xlabel("Mass (AMU)", fontsize=18, labelpad=10)
plt.ylabel("Normalized Intensity (a.u.)", fontsize=18, labelpad=10)

# Bigger title
plt.title("King CRAB Gas Content Before and After Cooling", fontsize=20, pad=15)

# Slightly expanded y-range so peaks don’t clip
plt.ylim(floor, 2)

# Legend
plt.legend(frameon=False, fontsize=14)

# Grid (subtle but visible)
plt.grid(True, which="both", linestyle="--", alpha=0.25)

# Make axes thicker and ticks larger
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.tick_params(
    axis='both',
    which='major',
    labelsize=14,
    width=2,
    length=8
)
plt.tick_params(
    axis='both',
    which='minor',
    width=1.5,
    length=4
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(DataWarm_BS["Mass"], DataWarm_BS["Intensity"],label="Before Cooling (bg-sub, norm)", color="Red")
plt.plot(DataCold_BS["Mass"], DataCold_BS["Intensity"],label="After Cooling (bg-sub, norm)", color="Blue")
plt.xlabel("Mass (AMU)")
plt.xlim(63,70)
plt.ylabel("Normalized Intensity (a.u.)")
plt.ylim(1e-5,.1)
plt.title("Background-Subtracted & Area-Normalized Spectra")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(DataWarm_BS["Mass"], DataWarm_BS["Intensity"],label="Before Cooling (bg-sub, norm)", color="Red")
plt.plot(DataCold_BS["Mass"], DataCold_BS["Intensity"],label="After Cooling (bg-sub, norm)", color="Blue")
plt.xlabel("Mass (AMU)")
plt.xlim(126,137)
plt.ylabel("Normalized Intensity (a.u.)")
plt.ylim(1e-5,.01)
plt.title("Background-Subtracted & Area-Normalized Spectra")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# My overall Xenon percent was confusing so I wanted to validate my analysis by redoing Krishans of Alpha 0 using 
# The same data set up by technique. I find that I agree with his anlaysis (he got 2.08% I get 2.06%) I believe
# The 10% Xe is due to multiple cycles gets the Xe near the bottle (and therefore the RGA) but I never circulate so it slowly gets back to 
# CRAB but not well thusly making subsquent captures easier

import numpy as np, pandas as pd

background_files = ["Background5_0.5AMUperSec.txt"]
leak_files       = ["LeakTest2_0.5AMUperSec.txt"]
leak_names       = ["LeakTest2"]  # match length to leak_files

configure_module(rga_helpers, globals())






Backgrounds = [read_rga(p) for p in background_files]
Leaks       = [(name, read_rga(p)) for name,p in zip(leak_names, leak_files)]

WINDOWS = {
    "Ar_Single":  (36.0, 41.0),
    "Ar_Double":  (19.0, 21.0),
    "Xe_Double":  (62.0, 70.0),     
    "Xe_Single":  (124.0, 136.5),}

for name, df in Leaks:
    df_bs = subtract_background(df, Backgrounds)
    Parts, Frac, FracPair = compositions(df_bs, WINDOWS)

    Xe   = Parts["Xe"]
    Ar   = Parts["Ar"]
    Tot  = Parts["Total"]

    Frac_Xe = 100*Xe/Tot if Tot>0 else 0.0
    Frac_Ar = 100*Ar/Tot if Tot>0 else 0.0
    Frac_Ot = 100*Parts["Other"]/Tot if Tot>0 else 0.0

    print(f"\n=== {name} ===")
    print(f"With Other:   Xe={Frac_Xe:.3f}%  Ar={Frac_Ar:.3f}%  Other={Frac_Ot:.3f}%  (sum={(Frac_Xe+Frac_Ar+Frac_Ot):.3f}%)")
    print(f"Xe vs Ar only: Xe={FracPair['Xe']:.3f}%  Ar={FracPair['Ar']:.3f}%")